# HCZ DAS View Tutorial and User Manual

HCZ DAS View is a **DAS Viewer / DAS Analysis** package. It focuses on reading DAS data, displaying metadata, visualizing time-channel data, plotting waveform/spectrum/PSD/spectrogram/FK views, running common preprocessing and analysis, and supporting a maintainable GUI workflow.

It is not a surface-wave inversion, MASW, F-J, dispersion-picking, earthquake-location, or source-location package. Event outputs in this manual are **event candidates** for DAS data inspection, not location or interpretation results.

## 1. Software Role: DAS Viewer / DAS Analysis

The package is intended for practical DAS data inspection and analysis:

- file reading and metadata summary;
- waterfall and waveform display;
- preprocessing and filtering;
- spectrum, PSD, spectrogram, statistics, spectral attributes, FK visualization;
- envelope, STA/LTA, and event candidate tables;
- CLI and GUI workflows.

## 2. DAS Data Shape: time x channel

All internal arrays follow:

```text
data.shape == (n_samples, n_channels)
```

Axis convention:

- `axis=0`: time samples;
- `axis=1`: channel / space axis.

Readers convert external file orientation into this convention before data reaches analysis functions.

In [ ]:
import numpy as np

sample_rate_hz = 1000.0
t = np.arange(1024) / sample_rate_hz
synthetic = np.column_stack([
    np.sin(2 * np.pi * 10.0 * t),
    0.5 * np.sin(2 * np.pi * 30.0 * t),
])
synthetic.shape

## 3. Reading Data and Metadata

Use reader-independent services for application code. A typical file workflow starts from metadata and bounded selections rather than full-file reads.

```python
from das_view.io.data_service import create_preview, read_selection

preview = create_preview("sample.h5", max_samples=2000, max_channels=512)
selection = read_selection("sample.h5", time_slice=slice(0, 4096), channel_slice=slice(0, 128))
metadata = selection.das_data.metadata
```

Use local paths in your own session; do not put private data paths in shared notebooks or documentation.

## 4. Waterfall and Waveform Views

A waterfall view shows the DAS matrix as a time-channel image. A waveform view shows one or more channels as traces. These are inspection tools for signal quality, clipping, time windows, active channels, and obvious events.

CLI examples:

```bash
python examples/validate_file.py sample.h5
python examples/plot_waveform.py sample.h5 --channel 0 --output waveform.png
```

## 5. Common Preprocessing

Stable preprocessing operations include:

- `demean`: subtract mean along a selected axis;
- `detrend`: remove linear trend;
- `taper`: reduce edge effects before spectral analysis;
- `normalize`: max-absolute, min-max, or standard-score scaling;
- `clip`: limit extreme amplitudes.

Preprocessing should be documented in analysis outputs because it changes measured attributes.

In [ ]:
from das_view.processing.preprocess import demean, normalize

processed = demean(synthetic, axis=0)
processed = normalize(processed, method="maxabs")
processed.shape

## 6. Common Filtering

Filtering tools include lowpass, highpass, bandpass, bandstop/notch-style filters through SciPy-backed implementations. Always check sample rate and Nyquist frequency before selecting cutoff frequencies.

```python
from das_view.processing.filters import bandpass

filtered = bandpass(data, sample_rate_hz=1000.0, low_hz=5.0, high_hz=80.0, axis=0)
```

## 7. Spectrum, PSD, and Spectrogram

Spectrum tools describe frequency content.

- Amplitude spectrum estimates frequency-domain amplitude.
- PSD estimates power per frequency using periodogram or Welch methods.
- Spectrogram shows time-varying frequency content.

Use bounded selections for large DAS files.

```bash
python examples/spectrum_file.py sample.h5 --channel 0
python examples/spectrum_file.py sample.h5 --channel 0 --psd welch
python examples/spectrum_file.py sample.h5 --channel 0 --spectrogram
```

## 8. Statistical Attributes

Basic statistics summarize amplitude behavior globally, time-wise, or channel-wise.

Key formulas:

```text
RMS = sqrt(mean(x^2))
Energy = sum(x^2)
Peak-to-peak = max(x) - min(x)
```

Supported attributes include mean, standard deviation, RMS, min, max, median, percentiles, absolute mean, peak-to-peak, energy, and finite/NaN/Inf summaries.

```bash
python examples/statistics_file.py sample.h5
python examples/statistics_file.py sample.h5 --axis time
python examples/statistics_file.py sample.h5 --axis channel
```

## 9. Spectral Attributes

Spectral attributes describe frequency-domain behavior for each channel or an averaged selection.

Key formulas:

```text
Band energy = sum P(f), f in [f1, f2]
Spectral centroid = sum(f P(f)) / sum(P(f))
Spectral bandwidth = sqrt(sum((f - centroid)^2 P(f)) / sum(P(f)))
```

Additional attributes include dominant frequency, peak spectral power, rolloff frequency, and band energy ratio.

```bash
python examples/spectral_attributes_file.py sample.h5 --bands 1 5 5 20
python examples/spectral_attributes_file.py sample.h5 --attributes --frequency-range 1 80
```

## 10. FK Visualization and FK-domain Smoke Filtering

FK tools inspect 2-D DAS wavefield content in frequency-wavenumber space. Apparent velocity is related to frequency and wavenumber:

```text
FK apparent velocity = |f / k|
```

FK-domain smoke filtering can suppress or preserve broad velocity ranges for DAS 2-D wavefield inspection. It is not a surface-wave imaging, inversion, MASW, F-J, or dispersion-picking workflow.

```bash
python examples/fk_file.py sample.h5 --time-start 0 --time-stop 4096 --channel-start 0 --channel-stop 512
python examples/fk_filter_file.py sample.h5 --time-start 0 --time-stop 4096 --channel-start 0 --channel-stop 512 --vmin 300 --vmax 3000
```

## 11. Envelope, STA/LTA, and Event Candidates

Event candidate tools highlight amplitude or energy changes in DAS data. They produce candidate intervals and channel indices, not source locations.

Key formulas:

```text
Envelope = |Hilbert(x)|
Short-window energy = moving_sum(x^2, STA window)
Long-window energy = moving_sum(x^2, LTA window)
STA/LTA = short_window_energy / long_window_energy
```

A threshold crossing creates an event candidate with start/end sample, duration, channel range, peak sample/channel, peak value, mean value, and score. Interpret candidates with context and visual review.

In [ ]:
from das_view.analysis.events import amplitude_envelope, sta_lta_ratio, detect_threshold_events

feature = amplitude_envelope(synthetic).values
ratio = sta_lta_ratio(synthetic, sta_samples=20, lta_samples=100).ratio
candidates = detect_threshold_events(feature, threshold=0.8, max_events=5).candidates
len(candidates)

## 12. CLI User Guide

Common CLI workflows:

```bash
python examples/validate_file.py sample.h5
python examples/statistics_file.py sample.h5 --output stats.json
python examples/spectral_attributes_file.py sample.h5 --bands 1 5 5 20 --output bands.json
python examples/event_detection_file.py sample.h5 --method stalta --sta 50 --lta 500 --trigger-on 3.0
python examples/event_detection_file.py sample.h5 --method envelope --threshold 0.8 --output events.csv
```

For large files, use `--time-start`, `--time-stop`, `--channel-start`, and `--channel-stop` to keep reads bounded.

## 13. GUI User Guide

Typical GUI workflow:

1. Launch the GUI with `python examples/run_gui.py` or the installed console script.
2. Open a supported DAS file.
3. Review metadata and the Waterfall preview.
4. Use Waveform, Spectrum, FK, and Analysis tabs for bounded inspection tasks.
5. Use Analysis-tab export buttons when a table or summary should be saved as JSON / CSV.

The GUI calls shared data, analysis, plotting, and export services. It does not expose format-internal HDF5 or DAT paths to the user workflow.


## 14. Current Limits and Interpretation Boundaries

Use this package as a DAS viewer and analysis toolkit. Current boundaries:

- Event candidate tables are screening aids, not earthquake locations or source-location results.
- FK visualization and FK-domain filtering are wavefield inspection tools, not specialized inversion workflows.
- Results depend on preprocessing, selected time/channel window, sample rate, channel spacing, thresholds, and noise conditions.
- Always inspect candidates visually and compare before/after preprocessing when analysis affects decisions.

## 15. ROI: time-channel window

A Region of Interest (ROI) is a half-open time-channel window for DAS review:

```text
time samples: [start_sample, end_sample)
channels: [channel_start, channel_end)
```

Useful ROI quantities include:

```text
duration_samples = end_sample - start_sample
n_channels = channel_end - channel_start
```

Time-only ROIs can omit the channel range. ROI records are analysis aids for selecting data windows; they are not location or interpretation results.

In [ ]:
from das_view.analysis.roi import TimeChannelROI

roi = TimeChannelROI(
    roi_id="roi_001",
    start_sample=100,
    end_sample=300,
    channel_start=2,
    channel_end=6,
    label="candidate_window",
    score=0.85,
)
roi.duration_samples, roi.n_channels

## 16. Annotation: label / description / category

Annotations attach human or workflow labels to an ROI. A minimal annotation has a required label and can also include a description, category, confidence, creator, and metadata.

Confidence, when used, is a value in `[0, 1]`. Avoid storing private paths, sensitive field names, or personal information in annotation metadata.

In [ ]:
from das_view.analysis.roi import Annotation

annotation = Annotation(
    annotation_id="ann_001",
    roi_id="roi_001",
    label="review",
    description="Candidate interval for visual review",
    category="event_candidate",
    confidence=0.7,
)
annotation.to_dict()

## 17. From Event Candidates to ROI

Event candidates can be converted into ROIs for review, plotting, and export. Padding can expand the time or channel window while keeping starts nonnegative.

This conversion preserves candidate score, but it still represents a data-analysis window only. It does not create a source location or geologic interpretation.

In [ ]:
from das_view.analysis.events import EventCandidate
from das_view.analysis.roi import rois_from_event_candidates

candidate = EventCandidate(
    event_id=1,
    start_sample=120,
    end_sample=180,
    duration_samples=60,
    channel_start=3,
    channel_end=3,
    peak_sample=150,
    peak_channel=3,
    peak_value=4.2,
    mean_value=1.1,
    max_value=4.2,
    score=4.2,
)
roi_set = rois_from_event_candidates([candidate], padding_samples=20, padding_channels=1)
roi_set.to_dict()

## 18. ROI Statistics and Spectral Attributes

ROI analysis reads each ROI as a bounded selection and then computes existing analysis attributes inside that window.

Typical statistics include RMS and energy:

```text
RMS_ROI = sqrt(mean(x_ROI^2))
Energy_ROI = sum(x_ROI^2)
```

Spectral ROI attributes reuse the same formulas as full-window spectral analysis, such as band energy and spectral centroid, but applied only to the ROI data.

```python
from das_view.analysis.service import compute_roi_statistics_for_file
summary = compute_roi_statistics_for_file("sample.h5", [roi])
```

## 19. JSON / CSV Export

ROI, annotation, event candidate, and analysis summary tables can be exported to JSON or CSV for downstream review.

CLI examples:

```bash
python examples/roi_export_file.py sample.h5 --roi 0 1000 0 100 --output-rois rois.json
python examples/roi_export_file.py sample.h5 --roi 0 1000 0 100 --output-summary roi_summary.csv
python examples/roi_export_file.py sample.h5 --detect-events --method envelope --threshold 0.8 --output-events events.csv --output-rois rois.json
```

Use placeholder paths in examples and local paths only in your private session.

In [ ]:
from das_view.io.export import rois_to_rows, annotations_to_rows

roi_rows = rois_to_rows(roi_set)
annotation_rows = annotations_to_rows([annotation])
roi_rows, annotation_rows

## 20. ROI and Event Candidate Interpretation Boundary

ROI and event candidate workflows are data-analysis aids. They help mark, summarize, compare, and export time-channel windows.

They are not:

- earthquake locations;
- source locations;
- geologic interpretations;
- inversion results;
- surface-wave imaging outputs.

Always inspect ROI windows visually, record preprocessing choices, and treat exported tables as review artifacts rather than final physical conclusions.

## 21. GUI Analysis panel

The Analysis tab provides a compact service-backed workflow for bounded DAS analysis. It is intended for review and screening, not for source location or geologic interpretation.

Common controls define a time-channel selection:

- `time_start`, `time_stop`, `time_step`;
- `channel_start`, `channel_stop`, `channel_step`;
- blank start/stop fields use the default bounded window.

The available analysis types are Statistics, Band energy, Spectral attributes, Event candidates - STA/LTA, Event candidates - Envelope threshold, and ROI statistics.


## 22. Statistics analysis operation

Choose **Statistics** to summarize a bounded selection. The axis option controls the reduction:

- `global`: one summary for the selected block;
- `time`: reduce along time and return one value per channel;
- `channel`: reduce along channel and return one value per sample.

Important formulas:

```text
RMS = sqrt(mean(x^2))
Energy = sum(x^2)
```

The summary reports count, finite count, NaN/Inf count, mean, standard deviation, RMS, energy, min, max, and requested percentiles.


## 23. Band energy / spectral attributes operation

Choose **Band energy** for frequency-band summaries such as `1-5,5-20,20-80`. Choose **Spectral attributes** for dominant frequency, centroid, bandwidth, rolloff, and total energy.

Key formulas:

```text
Band energy = sum P(f), f in [f1, f2]
Spectral centroid = sum(f P(f)) / sum(P(f))
Spectral bandwidth = sqrt(sum((f - centroid)^2 P(f)) / sum(P(f)))
```

Use channel averaging when a compact summary over the selected channel range is preferred.


## 24. Event candidate detection operation

The GUI exposes two event-candidate workflows:

- **Event candidates - STA/LTA**: set STA samples, LTA samples, trigger-on, optional trigger-off, minimum duration, merge gap, and maximum event count.
- **Event candidates - Envelope threshold**: set threshold, optional smoothing, minimum duration, merge gap, and maximum event count.

Key formulas:

```text
Envelope = |Hilbert(x)|
STA/LTA = short_window_energy / long_window_energy
```

The result table lists event candidates with sample range, channel range, peak sample/channel, peak value, and score. These are screening candidates only.


## 25. ROI statistics and JSON/CSV export

Choose **ROI statistics** to summarize one or more time-channel windows. Manual ROI rows use:

```text
start,end,ch_start,ch_end
```

For time-only ROI, use `start,end`. The GUI can also convert the latest event-candidate table into ROIs with optional sample/channel padding and maximum ROI count.

Use **Export JSON** for a summary payload and **Export CSV** for the displayed table rows. The export helpers create user-selected files; generated JSON/CSV outputs should be treated as local user artifacts.


## 26. GUI analysis interpretation boundary

GUI analysis results are bounded data summaries and review aids. Statistics, band energy, spectral attributes, event candidates, and ROI summaries should be interpreted together with acquisition context and domain expertise.

The Analysis tab does not perform earthquake location, source location, inversion, surface-wave imaging, MASW, F-J analysis, or dispersion picking. Event candidates and ROIs are not final geologic conclusions.


## 27. Installation

Install HCZ DAS View from the repository root with standard Python packaging commands. The distribution name is `hcz-das-view`, while the Python import package is `das_view`.

```bash
pip install .
pip install -e .[dev]
pip install -e .[gui]
pip install -e .[packaging]
```

Use the `gui` extra for the optional PyQt5 application, and the `packaging` extra when preparing local build artifacts. An editable install (`pip install -e ...`) is useful while working from a local checkout because command-line entry points continue to use the source tree.

A clean local environment check can install the package metadata without dependencies:

```powershell
python -m venv .tmp_release_venv
.tmp_release_venv\Scripts\python -m pip install -e . --no-deps
.tmp_release_venv\Scripts\python -m pip show hcz-das-view
```

The `--no-deps` form checks package installation only. Running analysis commands still requires runtime dependencies such as numpy and scipy.


## 28. Command-line entry points

After installation, the package provides these installed commands:

```bash
hcz-das-validate --help
hcz-das-preview --help
hcz-das-stats --help
hcz-das-spectrum --help
hcz-das-events --help
```

Typical usage with placeholder paths:

```bash
hcz-das-validate sample.h5
hcz-das-preview sample.h5 --output preview.png
hcz-das-stats sample.h5 --axis global --output stats.json
hcz-das-spectrum sample.h5 --channel 10 --mode welch --output welch.png
hcz-das-events sample.h5 --method stalta --sta 50 --lta 500 --trigger-on 3.0 --output events.csv
```

These commands use bounded service-layer workflows. Generated images, JSON files, and CSV files are local user artifacts.

Installed CLI commands are package entry points. The scripts under `examples/` are readable workflow examples for a source checkout; they demonstrate the same service-layer usage with placeholder paths.


## 29. GUI launch

Launch the optional GUI after installing the GUI extra:

```bash
hcz-das-view
```

The compatibility command is also available:

```bash
das-view-gui
```

The GUI supports metadata preview, Waterfall, Waveform, Spectrum, FK, and Analysis tabs. The GUI calls service-layer APIs and does not expose HDF5 or DAT internals as the user workflow. On Windows, `python -m das_view.gui.app --help` is the clearest console help form if a GUI-script executable does not echo text in the active shell.


## 30. Windows packaging and exe usage

Windows packaging notes are stored in `packaging/README_windows_packaging.md`. The local helper script and PyInstaller spec are:

```text
packaging/build_windows.ps1
packaging/hcz_das_view.spec
```

A local Windows packaging workflow is:

```powershell
python -m pip install -e .[gui,packaging]
powershell -ExecutionPolicy Bypass -File packaging\build_windows.ps1
dist\hcz-das-view\hcz-das-view.exe --help
dist\hcz-das-view\hcz-das-view.exe
```

PyInstaller output depends on the local Python environment. The executable is unsigned unless a separate signing step is used, and local build folders, wheels, archives, and exe files should stay outside source control.


## 31. Release usage notes

Before using a build as a release candidate, check:

1. Version metadata is correct.
2. Local sample smoke validation has been run without adding data to source control.
3. Installed CLI commands show help and run on small bounded examples.
4. Example scripts show help and use placeholder paths in documentation.
5. The GUI help path is available; on Windows, `python -m das_view.gui.app --help` is the clearest console form.
6. The GUI launches and opens a small supported sample.
7. Wheel and source archive build artifacts are created only as local artifacts.
8. A clean local environment can install the package metadata before broader dependency validation.
9. Windows exe packaging is validated if an exe is being distributed.
10. README and this tutorial notebook describe the current stable user workflow.
11. No real DAS data, local path lists, generated images, JSON/CSV outputs, build folders, wheels, archives, local environments, or exe files are included in source control.

Release artifacts should be treated separately from the source repository. The core interpretation boundary remains the same: DAS Viewer / DAS Analysis results are data review aids, not location, inversion, or geologic conclusions.


## 32. Extensions and plugins

HCZ DAS View includes a lightweight extension metadata layer. It describes built-in readers, processing operations, analysis workflows, plotting helpers, and export helpers in a common format.

Most users do not need to write plugins. The extension list is mainly useful for checking what capabilities are available in an installed package:

```bash
hcz-das-extensions --list
hcz-das-extensions --kind reader
hcz-das-extensions --kind analysis --json
```

The extension list is descriptive. Existing commands and GUI workflows still call the stable reader, processing, analysis, plotting, and export services directly.


## 33. Future third-party extensions

Future advanced users may expose optional capabilities through Python entry points in the `das_view.plugins` group. Discovery is explicit and on demand, so simply importing `das_view` does not scan external packages, read data, or start GUI code.

Good extension candidates are package-level additions such as a new reader, a processing helper, an analysis helper, a plotting helper, or an export helper. Specialized interpretation workflows should remain clearly separated from the core DAS Viewer / DAS Analysis workflow.

Extension metadata should describe capability boundaries. It should not include real data paths, private paths, generated artifacts, or claims that event candidates, ROIs, or general analysis summaries are source locations or geologic conclusions.


## 34. Public API and stable usage

For ordinary scripts and notebooks, prefer the documented package-level modules:

```python
from das_view.io import create_preview, read_selection, read_trace
from das_view.processing import demean, bandpass, apply_preprocess
from das_view.analysis import basic_statistics, spectral_attributes, detect_events_for_file
from das_view.plotting import plot_waterfall, plot_waveform
from das_view.plugins import ExtensionMetadata, ExtensionRegistry
```

Stable public API includes core data containers, reader-independent IO services, documented preprocessing/filter helpers, analysis services, Matplotlib plotting helpers, lightweight plugin metadata, and installed CLI entry points. GUI internals, worker classes, parser helpers, concrete reader internals, and underscore-prefixed helpers should be treated as implementation details.

Public API is kept stable where practical. Experimental interfaces can evolve as real workflows mature.


## 35. Data shape convention

All internal DAS arrays use a fixed time-by-channel convention:

```text
data.shape == (n_samples, n_channels)
axis=0 -> time / sample axis
axis=1 -> spatial channel / fiber channel axis
```

Readers convert supported file formats into this convention. If a source format stores data in another orientation, the reader should preserve useful source-orientation notes in metadata while returning arrays in the internal convention.

This convention applies to previews, selections, traces, preprocessing, statistics, spectral attributes, event candidates, ROI statistics, waterfall plots, waveform plots, and FK smoke workflows.


## 36. CLI / GUI / plugin usage choices

Use installed CLI commands when you want repeatable file-level operations from a terminal:

```bash
hcz-das-validate sample.h5
hcz-das-stats sample.h5 --axis global
hcz-das-events sample.h5 --method stalta --sta 50 --lta 500 --trigger-on 3.0
hcz-das-extensions --list
```

Use the GUI when you want interactive metadata review, bounded previews, waveform inspection, Spectrum/FK views, or the Analysis panel. Use `hcz-das-view` to launch it after installing the GUI extra.

Use `das_view.plugins` and `hcz-das-extensions` to inspect extension metadata. Most users do not need to write plugins; the plugin layer is a lightweight boundary for future package-level additions.


## 37. Interpretation boundaries for event candidates, ROI, and FK

Event candidates are screening aids. They identify time/channel windows that satisfy simple envelope or STA/LTA criteria. They are not earthquake locations, source locations, inversion results, or geologic conclusions.

ROI and annotation tools help organize data review. ROI statistics summarize bounded time-channel windows; they do not decide what a physical source is or where it occurred.

FK views and FK-domain smoke filtering are DAS 2D wavefield inspection tools. They are not a dedicated surface-wave imaging, MASW, F-J, dispersion-picking, or inversion workflow.

Treat all analysis outputs as context for domain review, not as final interpretation by themselves.


## 38. Troubleshooting

Installation checks:

```bash
python -m pip show hcz-das-view
python -c "import das_view; print(das_view.__version__)"
```

Entrypoint checks:

```bash
hcz-das-validate --help
hcz-das-stats --help
hcz-das-extensions --help
python -m das_view.gui.app --help
```

If a Windows GUI-script command does not echo help text in the active shell, use `python -m das_view.gui.app --help` for a visible console help path.

If a temporary-directory permission error appears during local validation, set the temporary directory to an ignored folder inside the checkout before rerunning the command:

```powershell
$env:TMP=(Join-Path (Get-Location) '.tmp_validation')
$env:TEMP=$env:TMP
```

Use placeholder paths in examples and documentation. Keep real DAS files, generated images, JSON/CSV outputs, local path lists, build folders, wheels, archives, local environments, and executables out of source control.


## 39. DAS QC metrics

DAS QC summaries describe whether a bounded time-channel selection is healthy enough for review. The core per-channel metrics are RMS, standard deviation, absolute mean, energy, NaN fraction, Inf fraction, zero fraction, clipping fraction, spike count, dead/quiet/noisy/low-energy flags, and a quality score.

```python
from das_view.analysis import data_quality_report, channel_quality_rows

report = data_quality_report(das_data)
rows = channel_quality_rows(report)
```

QC metrics are data-quality indicators. They help decide which channels or windows deserve closer inspection, but they are not location, inversion, or geologic interpretation results.


## 40. Bad channel detection, noise floor, and SNR

Bad-channel detection combines simple, transparent indicators such as unusually low RMS, unusually high RMS, dead-channel standard deviation, NaN/Inf fraction, clipping fraction, and spike count.

```python
from das_view.analysis import detect_bad_channels, estimate_noise_floor, estimate_snr

bad_channels = detect_bad_channels(das_data)
noise = estimate_noise_floor(das_data, method="mad")
snr = estimate_snr(das_data, signal_window=(1000, 2000), noise_window=(0, 1000))
```

The default noise-floor method uses a median absolute deviation style estimate. SNR is a ratio of signal-window amplitude to noise-window amplitude, so window choices should match the acquisition and review task.


## 41. Multiband feature map

Multiband maps convert a bounded DAS selection into a time-window x channel x frequency-band feature cube. This is useful for quickly comparing low-, middle-, and high-frequency energy across time and space.

```python
from das_view.analysis import multiband_energy_map, spectral_attribute_map

bands = [(1, 5), (5, 20), (20, 80)]
energy_map = multiband_energy_map(das_data, bands=bands, window_samples=1024, step_samples=512)
attribute_maps = spectral_attribute_map(das_data, window_samples=1024, step_samples=512)
```

The spectral attribute maps summarize dominant frequency, spectral centroid, bandwidth, and rolloff per window and channel. They are feature maps for review and export, not dispersion picking or surface-wave imaging.


## 42. Local channel coherence

Local channel coherence estimates how similar adjacent or lagged channels are. It can help flag coupling changes, bad channels, or areas where spatial continuity is weak.

```python
from das_view.analysis import local_channel_coherence

coherence = local_channel_coherence(das_data, channel_lag=1, window_samples=1024, step_samples=512)
```

The result is a correlation-style coherence map over time windows and channel pairs. It is not a velocity estimate, source location, or interpretation result.


## 43. QC command-line workflow

Use `hcz-das-qc` for bounded QC and feature-map workflows from an installed package:

```bash
hcz-das-qc sample.h5 --quality-report --output qc_report.json
hcz-das-qc sample.h5 --bad-channels --output bad_channels.csv
hcz-das-qc sample.h5 --multiband 1 5 5 20 20 80 --window-samples 1024 --step-samples 512 --output multiband.json
hcz-das-qc sample.h5 --coherence --channel-lag 1 --output coherence.json
```

Generated JSON and CSV files are local user artifacts. Keep them outside source control unless a separate dataset policy explicitly says otherwise.


## 44. Five-level DAS Analysis roadmap

The long-term DAS Analysis roadmap is organized by risk and generality:

1. **Level 1: DAS data quality / QC analysis** ? bad/dead/noisy channels, clipping, spikes, NaN/Inf/zero fractions, noise floor, SNR, quality scores, and QC report export.
2. **Level 2: Multiband / multispectral feature maps** ? time-window x channel x band energy, band ratios, dominant frequency, centroid, bandwidth, and rolloff maps.
3. **Level 3: Cross-channel coherence / spatial continuity** ? adjacent-channel correlation, local coherence, lagged-channel coherence, and windowed continuity scores.
4. **Level 4: Traditional robust denoising / enhancement** ? common-mode removal, median filtering, despike, channel balancing, local normalization, robust clipping, and FK-domain polish. This level is planned but not part of the current stable workflow.
5. **Level 5: Wavefield decomposition / apparent moveout assisted analysis** ? apparent slope or velocity attributes and directional energy summaries. This level is planned only.

Level 4 and Level 5 remain deferred. None of these levels turn HCZ DAS View into a MASW, F-J, dispersion-picking, source-location, inversion, or geologic interpretation package.
